# Main Code for NYTC Qualification
## 3rd Challenge

1) Recognize Pose and moves
2) Stops in box
3) Rotates and looks at correct face
4) moves to correct face and centralizes
5) moves to box B (in front of face)

## Setup

In [11]:
import importlib
import time
import cv2
import numpy as np
from ugot import ugot
import pose_yolo
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.


# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address.
# Change this string if your robot is on a different IP.
got.initialize("192.168.88.1")

# Load the built-in computer vision models that will be used later in the notebook.
# You can load additional models here if needed — just add the model name to the list.
got.load_models(
    [
        "color_recognition",  # detects dominant colors 
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode" # AprilTag recognition
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("Done!")

192.168.88.1:50051
Done!


## Recognize Pose and Move

#### Show Camera

In [4]:
got.open_camera()


def display_frame():
    frame = got.read_camera_data()
    if frame is not None:  # Checks if the frame is real!
        nparr = np.frombuffer(frame, np.uint8)  # Reads in the image data
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Turns numbers into color
        frame_rgb = cv2.cvtColor(data, cv2.COLOR_BGR2RGB)

        display(Image2.fromarray(frame_rgb))
        clear_output(wait=True)

In [5]:
try:
    while True:
        display_frame()

except KeyboardInterrupt:
    print("Done!")

Done!


#### Pose Recognition

In [1]:
try:
    print("Starting pose recognition...")
    run_pose_control_inline(
        robot_ip="192.168.88.1",
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=1,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
        got=got,
    )
except KeyboardInterrupt:
    print("Pose recognition stopped by user.")

Starting pose recognition...


NameError: name 'run_pose_control_inline' is not defined

## Rotate in Box
### Rotate and Find Face

#### Function Init

In [13]:
def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then approach them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # Phase 1: Spin and search
        while True:
            # Turn slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a name tag)
            name = got.get_words_result()

            # Check for any recognized faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective turn to center the robot on the target
                # turn=3 is clockwise; times=10, unit=2 means ~10 degree-units
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # Phase 2: Approach the target
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face; inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
                h = faces[0][3]  # Height of the face bounding box (proxy for distance)
                if h < height:
                    if c_x < 320 - gap:
                        # Face is too far LEFT — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is too far RIGHT — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centered but still small (far away) — move straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break  # Done

            clear_output(wait=True)

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

#### Face Find and Approach

In [9]:
TARGET_NAME = "Ryan"

In [14]:
try:
    print("Starting face recognition...")
    face_find_and_approach(
        gap=15,
        target_name=TARGET_NAME,
        turn_spd=10,
        strafe_spd=10,
        fwd_spd=10,
        height=120,
        adjust_turn=10,
    )
    got.mecanum_move_speed_times(
        direction=1, speed=20, times=15, unit=1
    )  # Go backwards a bit to not hit face
    got.mechanical_joint_control(angle1=0, angle2=-30, angle3=-30, duration=500)
    time.sleep(1)
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)
except KeyboardInterrupt:
    print("face recognition stopped by user.")

Reached Rvan!
face recognition stopped by user.
